# 4.1.6 Weight Initialization

Zero init, large random, Xavier, He — effect on gradient flow and convergence.

In [ ]:
import numpy as np

def make_weights(nin,nh,strat,seed=42):
    rng=np.random.default_rng(seed)
    if strat=='zeros':     return np.zeros((nin,nh)),np.zeros(1) # won't learn
    if strat=='large':     return rng.standard_normal((nin,nh))*10,rng.standard_normal((nh,1))*10
    if strat=='xavier':    lim=np.sqrt(6/(nin+nh)); return rng.uniform(-lim,lim,(nin,nh)),rng.standard_normal((nh,1))*np.sqrt(1/nh)
    if strat=='he':        return rng.standard_normal((nin,nh))*np.sqrt(2/nin),rng.standard_normal((nh,1))*np.sqrt(2/nh)

for s in ['zeros','large','xavier','he']:
    W,_=make_weights(2,32,s)
    print(f'{s:8s}: W.std={W.std():.4f}  W.max={np.abs(W).max():.4f}')

In [ ]:
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
sigmoid=lambda x:1/(1+np.exp(-np.clip(x,-500,500)))
relu=lambda x:np.maximum(0,x); relu_d=lambda x:(x>0).astype(float)

def train(X,y,W1,b1,W2,b2,lr=.05,ep=200):
    losses=[]
    for _ in range(ep):
        z1=X@W1+b1; a1=relu(z1); z2=a1@W2+b2; a2=sigmoid(z2)
        y2=y.reshape(-1,1); p=np.clip(a2,1e-7,1-1e-7)
        l=-np.mean(y2*np.log(p)+(1-y2)*np.log(1-p)); losses.append(l)
        n=len(y); dz2=(a2-y2)/n; dW2=a1.T@dz2; db2=dz2.sum(0)
        dz1=(dz2@W2.T)*relu_d(z1); dW1=X.T@dz1; db1=dz1.sum(0)
        W2-=lr*dW2; b2-=lr*db2; W1-=lr*dW1; b1-=lr*db1
    return losses

X,y=make_moons(500,noise=.2,random_state=42); Xs=StandardScaler().fit_transform(X)
Xt,_,yt,_=train_test_split(Xs,y,test_size=.2,random_state=42)

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(9,5))
for s in ['zeros','large','xavier','he']:
    W1,W2=make_weights(2,32,s); b1,b2=np.zeros(32),np.zeros(1)
    ls=train(Xt,yt,W1,b1,W2,b2); print(f'{s}: final={ls[-1]:.4f}')
    if np.isfinite(ls[-1]): ax.plot(ls,label=s)
ax.legend(); ax.set_title('Init Comparison'); ax.set_ylim(0,2); plt.show()